# Diabetes Risk Prediction — Model Comparison & Evaluation

**Goal:** Compare Random Forest and XGBoost against the logistic regression baseline,
then pick a final model based on the metrics that matter most for this problem.

Sections:
1. Recreate train/test split (same as notebook 02)
2. Baseline recap
3. Random Forest
4. XGBoost
5. Model comparison table
6. Threshold tuning
7. Select final model
8. Key takeaways

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score,
    precision_score, recall_score
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42

## 1. Recreate Train/Test Split

Same split and scaling as notebook 02, so results are directly comparable.

In [ ]:
from ucimlrepo import fetch_ucirepo

cdc_diabetes = fetch_ucirepo(id=891)
X_raw = cdc_diabetes.data.features
y_raw = cdc_diabetes.data.targets

df = pd.concat([X_raw, y_raw], axis=1)
df = df.drop_duplicates().reset_index(drop=True)

target_col = 'Diabetes_binary'
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

continuous_features = ['BMI', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']
continuous_features = [c for c in continuous_features if c in X_train.columns]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)

## 2. Baseline Recap (Logistic Regression)

Re-train the baseline here so all models and their metrics are in one notebook for easy comparison.

In [ ]:
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)

log_reg_pred = log_reg.predict(X_test_scaled)
log_reg_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

## 3. Random Forest

Tree-based models don't require scaling, so we use the unscaled features here.
Class imbalance handled with `class_weight='balanced'` again for a fair comparison.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, rf_pred, target_names=['No Diabetes', 'Diabetes/Pre']))

### Optional: Hyperparameter tuning for Random Forest

Uncomment to run a randomized search. This can take a few minutes on the full dataset —
consider running on a sample first if you're short on time.

In [ ]:
# param_dist = {
#     'n_estimators': [200, 300, 500],
#     'max_depth': [8, 12, 16, None],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4]
# }
#
# rf_search = RandomizedSearchCV(
#     RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
#     param_distributions=param_dist,
#     n_iter=10,
#     scoring='roc_auc',
#     cv=3,
#     random_state=RANDOM_STATE,
#     n_jobs=-1
# )
# rf_search.fit(X_train, y_train)
# print(rf_search.best_params_)
# rf_model = rf_search.best_estimator_

## 4. XGBoost

For imbalanced classification, use `scale_pos_weight` (ratio of negative to positive samples)
instead of `class_weight`.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, xgb_pred, target_names=['No Diabetes', 'Diabetes/Pre']))

## 5. Model Comparison Table

This table is the centerpiece of your README's results section.

In [ ]:
def get_metrics(name, y_true, y_pred, y_proba):
    return {
        'Model': name,
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

results = pd.DataFrame([
    get_metrics('Logistic Regression', y_test, log_reg_pred, log_reg_proba),
    get_metrics('Random Forest', y_test, rf_pred, rf_proba),
    get_metrics('XGBoost', y_test, xgb_pred, xgb_proba),
]).set_index('Model').round(3)

results

In [ ]:
# ROC curves — all models on one plot
for name, proba in [
    ('Logistic Regression', log_reg_proba),
    ('Random Forest', rf_proba),
    ('XGBoost', xgb_proba)
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

## 6. Threshold Tuning

The default 0.5 probability threshold isn't necessarily optimal for imbalanced problems.
Since missing an at-risk patient (false negative) is costlier than a false alarm, consider
lowering the threshold to boost recall — at the cost of some precision.

In [ ]:
# Use the best-performing model's probabilities (update variable name as needed)
best_proba = xgb_proba

precisions, recalls, thresholds = precision_recall_curve(y_test, best_proba)

plt.plot(thresholds, precisions[:-1], label='Precision')
plt.plot(thresholds, recalls[:-1], label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision & Recall vs. Threshold')
plt.legend()
plt.show()

In [ ]:
# Try a lower threshold, e.g. 0.35, and see the effect on metrics
custom_threshold = 0.35
custom_pred = (best_proba >= custom_threshold).astype(int)

print(f'Threshold = {custom_threshold}')
print(classification_report(y_test, custom_pred, target_names=['No Diabetes', 'Diabetes/Pre']))

## 6b. Matched-Recall Comparison

At the default 0.5 threshold, the three models aren't directly comparable on precision
because they're operating at different recall levels. To make a fair comparison, we find
the threshold for each model that produces the SAME recall (matching XGBoost's ~0.775),
then compare precision at that matched point. This answers: 'If all models had to catch
the same number of at-risk patients, which one does it with fewer false alarms?'

In [ ]:
def threshold_for_target_recall(y_true, y_proba, target_recall):
    """Find the highest threshold that achieves at least target_recall.
    A higher threshold at the same recall means fewer false positives (better precision)."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    # precision_recall_curve returns thresholds of length n-1 vs precisions/recalls of length n
    valid = recalls[:-1] >= target_recall
    if not valid.any():
        return 0.0  # target recall not achievable; lowest threshold gives max recall
    # Among thresholds achieving target recall, pick the highest threshold (best precision)
    return thresholds[valid].max()


def metrics_at_threshold(y_true, y_proba, threshold):
    y_pred_t = (y_proba >= threshold).astype(int)
    return {
        'Threshold': round(threshold, 3),
        'Precision': round(precision_score(y_true, y_pred_t), 3),
        'Recall': round(recall_score(y_true, y_pred_t), 3),
        'F1': round(f1_score(y_true, y_pred_t), 3)
    }


# Target recall = XGBoost's recall at its default threshold (from the comparison table)
target_recall = recall_score(y_test, xgb_pred)
print(f'Target recall (XGBoost @ 0.5): {target_recall:.3f}\n')

matched_results = {}
for name, proba in [
    ('Logistic Regression', log_reg_proba),
    ('Random Forest', rf_proba),
    ('XGBoost', xgb_proba)
]:
    t = threshold_for_target_recall(y_test, proba, target_recall)
    matched_results[name] = metrics_at_threshold(y_test, proba, t)

matched_df = pd.DataFrame(matched_results).T
matched_df

**How to read this table:** each model now has its threshold adjusted so recall is held
roughly constant across all three (matched to XGBoost's default-threshold recall). Whichever
model has the highest precision in this table is genuinely better at this recall level — not
just an artifact of comparing models at different operating points. Use this table, not the
default-threshold table, as the deciding comparison in your write-up.

## 7. Select Final Model

Based on the comparison table and threshold analysis above, document your choice and
reasoning here. Consider: Is a small drop in precision worth catching more at-risk patients?
What threshold makes sense for the intended use case (e.g. a screening tool)?

In [ ]:
# import joblib
# joblib.dump(xgb_model, '../src/final_model.pkl')
# print('Model saved.')

## 8. Key Takeaways

*(Fill this in — which model won, by how much, and why. e.g. 'XGBoost outperformed the
baseline with an ROC-AUC of X vs Y, and at a threshold of Z we catch W% of at-risk patients
while keeping false positives manageable.')*

- 
- 
- 

**Next notebook:** `04_model_interpretation_shap.ipynb` — explain what's driving the final
model's predictions using SHAP values.